In [1]:
import sys
from IPython.display import display, Javascript

def restart_kernel():
    """Restart the Jupyter Notebook kernel to reflect changes in modules and packages."""
    display(Javascript("Jupyter.notebook.kernel.restart()"))
    print("Kernel is restarting...")

restart_kernel()

import numpy as np
import torch
import tensorly as tl
from tensorly.decomposition import parafac
import sparse
import pickle
import os
import numpy.random as rn
import scipy.stats as st
from tensorly.cp_tensor import CPTensor

from bptf import BPTF as BPTF
import bptf

# Please run everything up to the first bptf fit
# Thereafter, run each following cell on their own

<IPython.core.display.Javascript object>

Kernel is restarting...


# Helper functions

In [2]:
def generate(shp=(30, 30, 20, 10), K=5, alpha=0.1, beta=0.1):
    """Generate a count tensor from the BPTF model.

    PARAMS:
    shp -- (tuple) shape of the generated count tensor
    K -- (int) number of latent components
    alpha -- (float) shape parameter of gamma prior over factors
    beta -- (float) rate parameter of gamma prior over factors

    RETURNS:
    Mu -- (np.ndarray) true Poisson rates
    Y -- (np.ndarray) generated count tensor
    """
    Theta_DK_M = [rn.gamma(alpha, 1./beta, size=(D, K)) for D in shp]
    Mu = tl.cp_to_tensor(CPTensor((None, Theta_DK_M)))
    assert Mu.shape == shp
    Y = rn.poisson(Mu)
    return Mu, Y

# Load data

In [3]:
use_existing_data = False

# for the first bptf.fit:
# using seed 100 causes the assert delta >= 0 to trip
# but using seed 0 doesn't
# please swap between the 2 seeds to get the 2 different types of assertion errors in the other cells
rn.seed(100)
num_months = 12

if use_existing_data:
    assert os.path.exists('sptensor.pkl'), 'No such file.'
    with open('sptensor.pkl', 'rb') as f:
        data = pickle.load(f)
    data = data[:, :, :, :num_months, :]
    data = sparse.COO(data)
else:
    data = generate(shp=(200, 200, 20, num_months, 3), K=10)[1]
    data = sparse.COO(data)

n_components = 10
max_iter = 500
tol = 1e-6

mask_flip = False

# Building mask (base example)

In [6]:
mask = np.zeros(data.shape)
# april of GDELT is set to missing
mask[:, :, :, 3, 1] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

if mask_flip:
    mask = (1 - mask.todense()).astype(np.int64)
    mask = sparse.COO(mask)

# Fit model

In [7]:
BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=0 if mask_flip else 1, tol=tol)

  0%|          | 0/500 [00:00<?, ?it/s]

mode 0
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 128993.65303119044
alpha * beta = 0.09893741647391313
Number of nonpositive rate parameters = 0
Smallest rate element = 128993.75196860691
mode 1
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 133416.85516768694
alpha * beta = 0.09854154797801312
Number of nonpositive rate parameters = 0
Smallest rate element = 133416.95370923492
mode 2
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 1195060.5408556983
alpha * beta = 0.09925463471488502
Number of nonpositive rate parameters = 0
Smallest rate element = 1195060.640110333
mode 3
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 1239483.5189

  0%|          | 1/500 [00:07<1:01:26,  7.39s/it]

mode 0
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 89870.72297047089
alpha * beta = 0.0869449402973615
Number of nonpositive rate parameters = 0
Smallest rate element = 89870.80991541118
mode 1
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 102128.92930488165
alpha * beta = 0.09965841463549023
Number of nonpositive rate parameters = 0
Smallest rate element = 102129.02896329628
mode 2
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 1016366.9432258105
alpha * beta = 0.10098365226139477
Number of nonpositive rate parameters = 0
Smallest rate element = 1016367.0442094628
mode 3
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 727189.9161373

  0%|          | 2/500 [00:14<58:20,  7.03s/it]  

...

mode 0
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 5161.8640744183285
alpha * beta = 0.10394292816490597
Number of nonpositive rate parameters = 0
Smallest rate element = 5161.968017346493
mode 1
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 4744.175672166764
alpha * beta = 0.1147823152574546
Number of nonpositive rate parameters = 0
Smallest rate element = 4744.290454482022
mode 2
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 54705.481454840985
alpha * beta = 0.10969511747706095
Number of nonpositive rate parameters = 0
Smallest rate element = 54705.59114995846
mode 3
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 5.29550015926361

  5%|▌         | 25/500 [02:58<56:26,  7.13s/it]

mode 0
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 5196.625241559729
alpha * beta = 0.10339590531252812
Number of nonpositive rate parameters = 0
Smallest rate element = 5196.728637465042
mode 1
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 4758.31371396606
alpha * beta = 0.11482047790681232
Number of nonpositive rate parameters = 0
Smallest rate element = 4758.428534443967
mode 2
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 54810.640648990804
alpha * beta = 0.10974917171785653
Number of nonpositive rate parameters = 0
Smallest rate element = 54810.75039816252
mode 3
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 6.778165698051453

  5%|▌         | 25/500 [03:05<58:42,  7.42s/it]


AssertionError: delta = -0.0001623136275350224

# Mask with last mode completely masked

In [5]:
mask = np.zeros(data.shape)
# april of GDELT is set to missing
mask[:, :, :, 3, :] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

if mask_flip:
    mask = (1 - mask.todense()).astype(np.int64)
    mask = sparse.COO(mask)

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=0 if mask_flip else 1, tol=tol)

  0%|          | 0/500 [00:00<?, ?it/s]

mode 0
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 118887.50989412489
alpha * beta = 0.099144977795355
Number of nonpositive rate parameters = 0
Smallest rate element = 118887.60903910268
mode 1
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 127805.84249692163
alpha * beta = 0.09869065968855914
Number of nonpositive rate parameters = 0
Smallest rate element = 127805.94118758132
mode 2
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 1223299.7395631452
alpha * beta = 0.10072429442248375
Number of nonpositive rate parameters = 0
Smallest rate element = 1223299.8402874395
mode 3
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = -2.8498470783

  0%|          | 1/500 [00:12<1:42:26, 12.32s/it]

mode 0
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 74684.34509277344
alpha * beta = 0.0815886350392303
Number of nonpositive rate parameters = 0
Smallest rate element = 74684.42668140848
mode 1
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 82991.42541503906
alpha * beta = 0.09944366223880197
Number of nonpositive rate parameters = 0
Smallest rate element = 82991.5248587013
mode 2
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 677695.9144897461
alpha * beta = 0.10216340147692085
Number of nonpositive rate parameters = 0
Smallest rate element = 677696.0166531475
mode 3
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = -3.3760443329811096

  0%|          | 2/500 [00:26<1:52:23, 13.54s/it]

mode 0
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 11756.0
alpha * beta = 0.0855865427310847
Number of nonpositive rate parameters = 0
Smallest rate element = 11756.085586542731
mode 1
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 9448.0
alpha * beta = 0.10705034035849233
Number of nonpositive rate parameters = 0
Smallest rate element = 9448.107050340359
mode 2
uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = 72002.0
alpha * beta = 0.1135164676694303
Number of nonpositive rate parameters = 0
Smallest rate element = 72002.11351646767
mode 3


C:\Users\luiyu\Documents\aaron_schein\bptf_new\bptf\src\bptf\bptf.py:260: RuntimeWarning: invalid value encountered in log
  self.G_DK_M[m] = np.exp(sp.psi(shp_DK) - np.log(rte_DK))
  0%|          | 2/500 [00:35<2:25:49, 17.57s/it]

uttkrp_DK.dtype = float64
mask.dtype = int64
self.E_DK_M.dtype = float64
masked uttkrp_DK.dtype = float64
Smallest value of uttkrp_DK = -8.731149137020111e-10
alpha * beta = 2.4088232723777873e-13
Number of nonpositive rate parameters = 2
Smallest rate element = -8.728740313747733e-10


AssertionError: 

# Mask with the 1st and 3rd indices masked

In [ ]:
mask = np.zeros(data.shape)
# april is set to missing
mask[:, :, :, 3, [0, 2]] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

if mask_flip:
    mask = (1 - mask.todense()).astype(np.int64)
    mask = sparse.COO(mask)

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=0 if mask_flip else 1, tol=tol)

  0%|          | 0/500 [00:00<?, ?it/s]

alpha * beta = 0.09865682141427463
Number of nonpositive shape parameters = 0
Smallest shape element = 7.417720486353945
Number of nonpositive rate parameters = 0
Smallest rate element = 238635.3581338652
alpha * beta = 0.09884390950492251
Number of nonpositive shape parameters = 0
Smallest shape element = 29.320886858761703
Number of nonpositive rate parameters = 0
Smallest rate element = 18219.463467163052
alpha * beta = 0.09914875541517927
Number of nonpositive shape parameters = 0
Smallest shape element = 3530.076042279061
Number of nonpositive rate parameters = 0
Smallest rate element = 195600.2461161263
alpha * beta = 0.09931330031906625
Number of nonpositive shape parameters = 0
Smallest shape element = 71.19265411108493
Number of nonpositive rate parameters = 0
Smallest rate element = 50816.82731742758
alpha * beta = 0.09728433860884539
Number of nonpositive shape parameters = 0
Smallest shape element = 262373.64075628493
Number of nonpositive rate parameters = 0
Smallest rate 

  0%|          | 1/500 [00:07<1:06:23,  7.98s/it]

alpha * beta = 1.3156603734846892
Number of nonpositive shape parameters = 0
Smallest shape element = 4.83724723451653
Number of nonpositive rate parameters = 0
Smallest rate element = 239960.22354439853
alpha * beta = 0.0989925862227331
Number of nonpositive shape parameters = 0
Smallest shape element = 10.389451055078247
Number of nonpositive rate parameters = 0
Smallest rate element = 16620.121668245287
alpha * beta = 0.09911154551668355
Number of nonpositive shape parameters = 0
Smallest shape element = 1071.2580798216013
Number of nonpositive rate parameters = 0
Smallest rate element = 144474.643146387
alpha * beta = 0.10057704165043309
Number of nonpositive shape parameters = 0
Smallest shape element = 17.47112625192181
Number of nonpositive rate parameters = 0
Smallest rate element = 31908.22222471071
alpha * beta = 0.09823499791389008
Number of nonpositive shape parameters = 0
Smallest shape element = 13541.992360319917
Number of nonpositive rate parameters = 0
Smallest rate el

  0%|          | 2/500 [00:16<1:07:21,  8.12s/it]

...

alpha * beta = 1.4669840498748714
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000731934684
Number of nonpositive rate parameters = 0
Smallest rate element = 2787.831304880402
alpha * beta = 0.11056233783762204
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000650439772
Number of nonpositive rate parameters = 0
Smallest rate element = 198.7679022814731
alpha * beta = 0.11294246052705144
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000281169053625
Number of nonpositive rate parameters = 0
Smallest rate element = 2440.81871351428
alpha * beta = 8.634016340370386e-11
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000004669281277
Number of nonpositive rate parameters = 0
Smallest rate element = 1.0858542010610695e-07
alpha * beta = 0.03949308347633759
Number of nonpositive shape parameters = 0
Smallest shape element = 0.6798632840116212
Number of nonpositive rate parameters = 0
Sma

  2%|▏         | 10/500 [01:21<1:06:33,  8.15s/it]

alpha * beta = 1.4704947461036424
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000743414378
Number of nonpositive rate parameters = 0
Smallest rate element = 2310.1350975827654
alpha * beta = 0.11083027067567369
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000650573912
Number of nonpositive rate parameters = 0
Smallest rate element = 164.57667014559155
alpha * beta = 0.11312240328548549
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000136015609182
Number of nonpositive rate parameters = 0
Smallest rate element = 2201.779353555959
alpha * beta = 7.104830565985153e-12
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000004500546161
Number of nonpositive rate parameters = 0
Smallest rate element = 2.612430870102077e-07
alpha * beta = 0.027019348428500085
Number of nonpositive shape parameters = 0
Smallest shape element = 0.15946039714778149
Number of nonpositive rate parameters = 0

  2%|▏         | 10/500 [01:29<1:12:46,  8.91s/it]


AssertionError: delta = -0.0004582024948243403

# Mask with only diagonals masked

In [ ]:
mask = np.zeros(data.shape)

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

if mask_flip:
    mask = (1 - mask.todense()).astype(np.int64)
    mask = sparse.COO(mask)

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=0 if mask_flip else 1, tol=tol)

100%|██████████| 500/500 [56:35<00:00,  6.79s/it]


BPTF(data_shape=(200, 200, 20, 24, 3), n_components=10)